# Quantum Conditional Boltzmann Machines on Real Financial Time Series
### A Scaling Study of the Quantum Advantage Gap

This notebook reproduces the full pipeline described in `docs/project_overview.docx`:

1. Mount Drive + install dependencies
2. Download & cache real market data for a diversified 7-asset basket
3. Build the fixed-eval-window / growing-training-prefix scaling splits
4. Train all four architectures (CRBM, QCRBM, QFeatureQRBM, QQRBM) across the
   (asset x train_size x seed) grid
5. Run Holm-corrected paired significance tests
6. Save results + figures back to Drive

**Run cells top to bottom.** Each stage checkpoints to Drive, so if your
Colab session disconnects you can re-run from where you left off (the grid
runner in Step 4 resumes automatically from its checkpoint CSV).


## Step 0 — Mount Drive & set up project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/qcrbm_project'  # <-- edit if you placed it elsewhere
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results/figures', exist_ok=True)

import sys
sys.path.insert(0, PROJECT_DIR)
print('Project dir:', PROJECT_DIR)


**First-time setup only:** if you haven't uploaded the project folder to Drive yet,
upload the unzipped `qcrbm_project/` folder (containing `src/`, `docs/`, `requirements.txt`)
to `MyDrive/qcrbm_project` via the Colab file browser or `drive.google.com` before
running the cell above.

## Step 1 — Install dependencies

In [ ]:
import os
os.environ['PROJECT_DIR'] = PROJECT_DIR
!pip install -q -r "$PROJECT_DIR/requirements.txt"
print('Dependencies installed.')


## Step 2 — Download & cache market data

In [ ]:
from src.data import (
    download_prices, compute_log_returns, adf_report, correlation_table,
    build_all_splits, regime_diagnostics_table, prepare_all,
    DEFAULT_TICKERS, DEFAULT_TRAIN_SIZES, DEFAULT_CONTEXT_LEN,
)

prices = download_prices(DEFAULT_TICKERS, cache_path=f'{PROJECT_DIR}/data/prices_raw.csv')
log_returns = compute_log_returns(prices)
log_returns.to_csv(f'{PROJECT_DIR}/data/log_returns.csv')

print('Assets:', list(log_returns.columns))
print('History range:', log_returns.index.min(), '->', log_returns.index.max())
print('Total observations per asset:', len(log_returns))


### Stationarity check (ADF) and asset-correlation table

Report both explicitly — the correlation table is the transparency step for
Drawback 1 (independence across assets is approximate, not exact).

In [ ]:
adf_df = adf_report(log_returns)
adf_df.to_csv(f'{PROJECT_DIR}/results/adf_report.csv')
print("ADF stationarity report (all assets should reject the unit-root null,")
print("i.e. p_value < 0.05, for log returns):")
adf_df


In [ ]:
corr = correlation_table(log_returns)
corr.to_csv(f'{PROJECT_DIR}/results/asset_correlation.csv')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im)
plt.title('Asset return correlation (report alongside independence claims)')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/figures/asset_correlation.png', dpi=150)
plt.show()


## Step 3 — Build the scaling-study splits (fixed eval window, growing training prefix)

In [ ]:
splits = build_all_splits(
    log_returns=log_returns,
    tickers=list(log_returns.columns),
    train_sizes=DEFAULT_TRAIN_SIZES,
)

diag = regime_diagnostics_table(splits)
diag.to_csv(f'{PROJECT_DIR}/results/regime_diagnostics.csv', index=False)
print('Regime / volatility diagnostics per (asset, train_size) -- report next to the')
print('scaling-curve RMSE table so any regime confound is visible (Drawback 2):')
diag


In [ ]:
prepared = prepare_all(splits, context_len=DEFAULT_CONTEXT_LEN)
print('Prepared datasets for:')
for ticker, by_size in prepared.items():
    print(f'  {ticker}: train sizes {sorted(by_size.keys())}')


## Step 4 — Train all four architectures across the full grid

This is the long-running step. It checkpoints to
`results/scaling_grid_results.csv` after every single (asset, train_size, seed, model)
run, so if the Colab session disconnects, just re-run this cell — it resumes
automatically from the checkpoint.

**Runtime note:** QQRBM and QFeatureQRBM are the slowest (per-sample circuit
evaluation). If time is tight, drop them from `models=[...]` below on your
first pass and add them back for a final run — the CRBM vs. QCRBM comparison
is the headline result and should be protected first.

In [ ]:
from src.train import run_scaling_grid, DEFAULT_SEEDS, DEFAULT_EPOCHS

results = run_scaling_grid(
    prepared,
    models=["CRBM", "QCRBM", "QFeatureQRBM", "QQRBM"],   # trim this list if runtime is tight
    seeds=DEFAULT_SEEDS,
    epochs=DEFAULT_EPOCHS,
    checkpoint_path=f'{PROJECT_DIR}/results/scaling_grid_results.csv',
)
results.tail(20)


## Step 5 — Statistical comparison (Holm-corrected paired tests per training size)

In [ ]:
from src.stats import compare_all_to_reference, power_analysis
import pandas as pd

results = pd.read_csv(f'{PROJECT_DIR}/results/scaling_grid_results.csv')

all_comparisons = []
for train_size in sorted(results['train_size'].unique()):
    sub = results[results['train_size'] == train_size]
    comp = compare_all_to_reference(
        sub, reference_model="CRBM",
        candidate_models=["QCRBM", "QFeatureQRBM", "QQRBM"],
        group_cols=("ticker", "seed"),
    )
    comp.insert(0, "train_size", train_size)
    all_comparisons.append(comp)

comparison_df = pd.concat(all_comparisons, ignore_index=True)
comparison_df.to_csv(f'{PROJECT_DIR}/results/significance_tests.csv', index=False)

n_per_comparison = results.groupby('train_size')['seed'].nunique().iloc[0] * results['ticker'].nunique()
print(f"Approx. n per comparison: {n_per_comparison}")
print(f"Minimum detectable effect size (power=0.8, alpha=0.05): {power_analysis(n=n_per_comparison):.3f}")
comparison_df


## Step 6 — Window-information-ceiling diagnostic (noise-floor check)

In [ ]:
from src.stats import window_information_ceiling

ceiling_rows = []
for ticker, by_size in prepared.items():
    ds = by_size[max(by_size.keys())]  # largest train size available
    ceiling = window_information_ceiling(ds.u_train, ds.v_train)
    ceiling_orig_scale = ceiling * ds.scaler_std
    ceiling_rows.append({"ticker": ticker, "nonlinear_window_floor_rmse": ceiling_orig_scale})

ceiling_df = pd.DataFrame(ceiling_rows)
ceiling_df.to_csv(f'{PROJECT_DIR}/results/window_information_ceiling.csv', index=False)
print("Compare these floors against your model RMSE tables -- if models cluster")
print("near this floor, a 'tie' between architectures may just be the noise floor,")
print("not genuine equivalence (base paper Sec 5.5).")
ceiling_df


## Step 7 — Scaling curves (main result figure)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 1, figsize=(8, 5))
summary = results.groupby(['model', 'train_size'])['rmse'].agg(['mean', 'std']).reset_index()

for model in summary['model'].unique():
    sub = summary[summary['model'] == model].sort_values('train_size')
    axes.errorbar(sub['train_size'], sub['mean'], yerr=sub['std'], marker='o', label=model, capsize=3)

axes.set_xscale('log')
axes.set_xlabel('Training-set size (log scale)')
axes.set_ylabel('Test RMSE (mean ± SD across assets & seeds)')
axes.set_title('Quantum vs. Classical Scaling Curve on Real Market Data')
axes.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/figures/scaling_curve.png', dpi=150)
plt.show()


## Done

All results are saved under `results/` on your Drive:
- `scaling_grid_results.csv` — every (asset, train_size, seed, model) RMSE
- `significance_tests.csv` — Holm-corrected paired comparisons per train_size
- `asset_correlation.csv`, `regime_diagnostics.csv`, `window_information_ceiling.csv` — diagnostics for Drawbacks 1 & 2
- `figures/scaling_curve.png`, `figures/asset_correlation.png` — main figures

Use these directly in the Results/Discussion sections of the paper (see
`docs/project_overview.docx` for the write-up structure and Week 2 writing plan).